## **Setup**

In [ ]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

if IS_KAGGLE:
    !pip install optuna
    !pip install fastparquet

Repo already exists — pulling latest changes
Already up to date.


In [ ]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

## **Imports**

In [ ]:
import os
import pandas as pd
import optuna
from xgboost import XGBRanker
from sklearn.metrics import roc_auc_score

from Challenge.paths import load_holdout_split, XGBOOST_DATAFRAMES
from Challenge.utils import evaluate_recommender
from Challenge.XGBoostReranker import XGBoostRerankerRecommender, ProgressCallback
from Challenge.hyper_tuning import ModelOptimizer

## **Load Data**

In [ ]:
_, URM_outer = load_holdout_split()
del _

In [ ]:
file_path = os.path.join("kaggle/input", "XG-boost-dfs", "training_OOF.parquet")

X_train = pd.read_parquet(
    file_path,
    engine='fastparquet'
)

print(f"Loaded successfully from: {file_path}")
X_train

In [ ]:
if not X_train['UserID'].is_monotonic_increasing:
    print("⚠️ Data is NOT sorted by UserID.")
    print("Sorting and overwriting file for future efficiency...")
    
    # Sort and reset index (critical for contiguous memory in ML)
    X_train = X_train.sort_values(by='UserID').reset_index(drop=True)
    
    # Overwrite the file on disk
    X_train.to_parquet(
        path=file_path,
        engine='fastparquet',
        compression='zstd',
        index=False
    )
    
    print(f"✅ Data sorted and saved to {file_path}")

In [ ]:
assert X_train['UserID'].is_monotonic_increasing, "UserID column is not sorted in increasing order!"

# Group Calculation
groups = X_train.groupby("UserID").size().values

# Target and Feature Matrix Definition
y_train = X_train["Label"]

# Drop the identifiers (UserID, ItemID) and the target (Label), keeping only the features
X_train = X_train.drop(columns=["Label", "UserID", "ItemID"], errors='ignore')

# Make categorical features if any
for col in ['User_Cluster', 'Item_Cluster', 'Cluster_Interaction']:
    X_train[col] = X_train[col].astype('category')

assert groups.sum() == X_train.shape[0], "Group sum does not match total data size!"

In [ ]:
file_path = os.path.join("kaggle/input", "XG-boost-dfs", "validation_OOF.parquet")

X_val = pd.read_parquet(
    file_path,
    engine='fastparquet'
)

# Make categorical features if any
for col in ['User_Cluster', 'Item_Cluster', 'Cluster_Interaction']:
    X_val[col] = X_val[col].astype('category')

# Reorder val columns to match training set
feature_cols = [c for c in X_train.columns if c != 'Label'] + ['UserID', 'ItemID']
X_val = X_val[feature_cols]

print(f"Loaded successfully from: {file_path}")
X_val

## **Optimize XG Boost hypeparameters**

In [ ]:
optimizer = ModelOptimizer("XG_Boost")

recommender = XGBoostRerankerRecommender(None, X_val) # Pre process X_val for faster evaluation

In [ ]:
STUDY_NAME = XGBoostRerankerRecommender.RECOMMENDER_NAME + "_v1"

In [ ]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        'learning_rate': optuna_trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'n_estimators': optuna_trial.suggest_int('n_estimators', 100, 3000),
        'max_depth': optuna_trial.suggest_int('max_depth', 4, 12),
        'reg_alpha': optuna_trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': optuna_trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'colsample_bytree': optuna_trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample': optuna_trial.suggest_float('subsample', 0.6, 1.0),
        
        'objective': optuna_trial.suggest_categorical('objective', ['rank:pairwise', 'rank:ndcg']),
        'grow_policy': 'depthwise'
    }

    # Instantiate XGB model with current hyperparameters
    XGB_model = XGBRanker(
        **params,
        
        # Use GPU for faster training
        device="cuda",
        
        random_state=42,
        callbacks=[ProgressCallback(params['n_estimators'], period=100)]
    )

    # Fit XG boost reranker
    XGB_model.fit(
        X_train,
        y_train,
        group=groups,
        verbose=False
    )

    # Evaluate training performance
    # y_pred_scores = XGB_model.predict(X_train)
    # auc_score = roc_auc_score(y_train, y_pred_scores)
    # print(f"Training AUC Score: {auc_score:.4f}")

    # Set the XGB model in the recommender
    recommender.XGB_model = XGB_model
    
    # Evaluate
    score = evaluate_recommender(recommender, at=20, URM_validation=URM_outer)
    print(f"Validation Score (RECALL@20): {score:.4f}")

    optimizer.log_folds([score], params)

    return score

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [ ]:
optuna_study.best_params